In [85]:
# !pip install aiomoex
# !pip install tinkoff-investments
# !pip install yfinance

Открытые вопросы:
* Стоит ли описывать варианты получения данных и причину выбора?

Что нужно сделать ближайшее:
* Выгрузить IMOEX индекс на периоде для сравнения (в идеале потом можно еще с открытием вкладов сравнить, но сложно кодом считать)
* Реализовать функции расчета индикаторов, которые необходимы будут для обучения


In [86]:
from tinkoff.invest import Client, AsyncClient, CandleInterval, SecurityTradingStatus
from tinkoff.invest.services import InstrumentsService
from tinkoff.invest.utils import quotation_to_decimal, now
from tinkoff.invest.caching.instruments_cache.instruments_cache import InstrumentsCache

import pandas as pd
from dotenv import load_dotenv

import os
# import asyncio
import time
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Optional
import warnings

warnings.filterwarnings("ignore")

In [87]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

In [88]:
load_dotenv()

# TOKEN = "REDACTED"
TOKEN = os.getenv("TOKEN")

In [89]:
tickers_to_test = ["SBER", "GLDRUB_TOM", "CNYRUB_TOM", "BRH6", "MOEX"]

Словарь настроек интервалов: (Константа API, Максимальный шаг скачивания)

In [90]:
INTERVALS = {
    "5min":  (CandleInterval.CANDLE_INTERVAL_5_MIN, timedelta(days=7)),
    "15min": (CandleInterval.CANDLE_INTERVAL_15_MIN, timedelta(days=21)),
    "1hour": (CandleInterval.CANDLE_INTERVAL_HOUR, timedelta(days=31)),
    "4hour": (CandleInterval.CANDLE_INTERVAL_4_HOUR, timedelta(days=31 * 2)),
    "1day":  (CandleInterval.CANDLE_INTERVAL_DAY, timedelta(days=365))
}

In [91]:
rates_data = {
    'Year': [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'MaxRate': [0.15325, 0.0999, 0.084, 0.07245, 0.07529, 0.05927, 0.04486, 0.07738, 0.0813, 0.14793, 0.21723]
}
inflation_data = {
    'Year': [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
    'MaxRate': [0.1291, 0.0538, 0.0252, 0.0427, 0.0305, 0.0491, 0.0839, 0.1192, 0.0742, 0.0951, 0.0526]
}
df_rates = pd.DataFrame(rates_data).set_index('Year')
df_inflation = pd.DataFrame(inflation_data).set_index('Year')

Подробная информация по тикеру.

**Не работает с индексами вроде индекса МосБиржи**

In [92]:
def get_info(ticker, TOKEN = TOKEN):
    with Client(TOKEN) as client:
        instruments: InstrumentsService = client.instruments
        tickers = []
        for method in ["shares", "bonds", "etfs", "currencies", "futures"]:
            for item in getattr(instruments, method)().instruments:
                tickers.append(
                    {
                        "name": item.name,
                        "ticker": item.ticker,
                        "class_code": item.class_code,
                        "figi": item.figi,
                        "uid": item.uid,
                        "type": method,
                        "min_price_increment": quotation_to_decimal(
                            item.min_price_increment
                        ),
                        "scale": 9 - len(str(item.min_price_increment.nano)) + 1,
                        "lot": item.lot,
                        "trading_status": str(
                            SecurityTradingStatus(item.trading_status).name
                        ),
                        "api_trade_available_flag": item.api_trade_available_flag,
                        "currency": item.currency,
                        "exchange": item.exchange,
                        "buy_available_flag": item.buy_available_flag,
                        "sell_available_flag": item.sell_available_flag,
                        "short_enabled_flag": item.short_enabled_flag,
                        "klong": quotation_to_decimal(item.klong),
                        "kshort": quotation_to_decimal(item.kshort),
                    }
                )

        tickers_df = pd.DataFrame(tickers)

        ticker_df = tickers_df[tickers_df["ticker"] == ticker]
        if ticker_df.empty:
            print("There is no such ticker: %s", ticker)
            # return

        figi = ticker_df["figi"].iloc[0]
        print(f"\nTicker {ticker} have figi={figi}\n")
        print(f"Additional info for this {ticker} ticker:")
        print(ticker_df.iloc[0])

        return figi

ticker = "SPY"
get_info(ticker)


Ticker SPY have figi=BBG000BDTBL9

Additional info for this SPY ticker:
name                                                   SPDR S&P 500 ETF Trust
ticker                                                                    SPY
class_code                                                              SPBXM
figi                                                             BBG000BDTBL9
uid                                      59ba0c48-4f13-429a-b1a1-6aff71baca05
type                                                                     etfs
min_price_increment                                                0.01000000
scale                                                                       2
lot                                                                         1
trading_status              SECURITY_TRADING_STATUS_NOT_AVAILABLE_FOR_TRADING
api_trade_available_flag                                                 True
currency                                                             

'BBG000BDTBL9'

Получить только FIGI.

In [93]:
def get_figi(ticker: str) -> str:
    ticker = ticker.upper()
    with Client(TOKEN) as client:
        response = client.instruments.find_instrument(query=ticker)
        
        # 1. Сначала ищем идеальный вариант: совпадение тикера + доступность для торгов
        for instrument in response.instruments:
            if instrument.ticker == ticker and instrument.api_trade_available_flag:
                # Маленький лайфхак: для акций РФ основной режим торгов - TQBR (акции) или SPBFUT (фьючерсы)
                if instrument.class_code in ['TQBR', 'TQCB', 'SPBFUT', 'CETS', 'TQTF']:
                    print(f"Подтверждено: {ticker} ({instrument.name}), FIGI: {instrument.figi}")
                    return instrument.figi
        
        # 2. Если идеальный не нашли, берем любой доступный с точным тикером
        for instrument in response.instruments:
            if instrument.ticker == ticker and instrument.api_trade_available_flag:
                return instrument.figi
                
        raise ValueError(f"Не удалось найти активный торговый инструмент для тикера {ticker}")

ticker = "SBER"
get_figi(ticker)

Подтверждено: SBER (Сбер Банк), FIGI: BBG004730N88


'BBG004730N88'

In [94]:
for t in tickers_to_test:
    try:
        figi = get_figi(t)
        print(f"Успех! {t} -> {figi}")
    except Exception as e:
        print(f"Ошибка для {t}: {e}")

Подтверждено: SBER (Сбер Банк), FIGI: BBG004730N88
Успех! SBER -> BBG004730N88
Подтверждено: GLDRUB_TOM (Золото), FIGI: BBG000VJ5YR4
Успех! GLDRUB_TOM -> BBG000VJ5YR4
Подтверждено: CNYRUB_TOM (Китайский юань), FIGI: BBG0013HRTL0
Успех! CNYRUB_TOM -> BBG0013HRTL0
Подтверждено: BRH6 (BR-3.26 Нефть Brent), FIGI: FUTBR0326000
Успех! BRH6 -> FUTBR0326000
Подтверждено: MOEX (Московская Биржа), FIGI: BBG004730JJ5
Успех! MOEX -> BBG004730JJ5


In [95]:
def get_5min(ticker: str) -> pd.DataFrame:
    ticker = ticker.upper()
    file_path = DATA_DIR / f"{ticker}_5min.parquet"

    # Если файл есть — читаем и сразу делаем DateTime колонкой
    if file_path.exists():
        print(f"Загружаю из кэша: {ticker}")
        df = pd.read_parquet(file_path)
        if df.index.name == "DateTime":
            df = df.reset_index()
        return df.sort_values("DateTime", ascending=False)

    # Если файла нет — скачиваем
    print(f"Скачиваю {ticker} с 01.01.2025...")
    figi = get_figi(ticker)
    new_rows = []
    current = datetime(2025, 1, 1)
    now = datetime.now()

    with Client(TOKEN) as client:
        while current < now:
            end = min(current + timedelta(days=7), now - timedelta(minutes=10))

            if end <= current:
                break

            for _ in range(3):
                try:
                    candles = client.market_data.get_candles(
                        figi=figi,
                        from_=current,
                        to=end,
                        interval=CandleInterval.CANDLE_INTERVAL_5_MIN
                    ).candles

                    for c in candles:
                        new_rows.append({
                            "DateTime": c.time,
                            "Open":  c.open.units  + c.open.nano  / 1e9,
                            "High":  c.high.units  + c.high.nano  / 1e9,
                            "Low":   c.low.units   + c.low.nano   / 1e9,
                            "Close": c.close.units + c.close.nano / 1e9,
                            "Volume": c.volume
                        })
                    print(f"  +{len(candles)} свечей → {end.strftime('%Y-%m-%d %H:%M')}")
                    break
                except Exception as e:
                    if "RESOURCE_EXHAUSTED" in str(e):
                        print("Лимит! Жду 45 сек...")
                        time.sleep(45)
                    else:
                        time.sleep(5)
            else:
                print(f"Не удалось скачать кусок {current.date()} → {end.date()}")
            
            current = end
            time.sleep(1.2)

    if not new_rows:
        raise RuntimeError(f"Не удалось скачать данные для {ticker}")

    df = pd.DataFrame(new_rows)
    df = df.sort_values("DateTime", ascending=False)
    
    df.to_parquet(file_path, compression="zstd", index=False)
    print(f"Сохранено: {file_path} → {len(df):,} строк")

    return df

In [96]:
def get_candles_data(ticker: str, 
                     interval_name: str = "5min",
                     start_date: datetime = datetime(2025, 1, 1), 
                     end_date: Optional[datetime] = None) -> pd.DataFrame:
    """
    Загружает исторические свечи, используя локальное кэширование в Parquet.
    Решает проблему смешивания tz-aware и tz-naive дат.
    """
    ticker = ticker.upper()
    if interval_name not in INTERVALS:
        raise ValueError(f"Неверный интервал. Доступные: {list(INTERVALS.keys())}")
    
    t_interval, chunk_step = INTERVALS[interval_name]
    file_path = DATA_DIR / f"{ticker}_{interval_name}.parquet"
    
    # 1. Нормализация дат (приводим всё к naive - без часовых поясов для сравнения)
    if start_date.tzinfo is not None:
        start_date = start_date.replace(tzinfo=None)
    
    if end_date is None:
        end_date = datetime.now()
    elif end_date.tzinfo is not None:
        end_date = end_date.replace(tzinfo=None)

    df = pd.DataFrame()

    # 2. Проверка локального кэша
    if file_path.exists():
        df = pd.read_parquet(file_path)
        # Очищаем кэшированные данные от TZ для корректного сравнения
        df['DateTime'] = pd.to_datetime(df['DateTime']).dt.tz_localize(None)
        
        if not df.empty:
            cache_min = df['DateTime'].min()
            cache_max = df['DateTime'].max()
            
            # Если кэш покрывает запрошенный период (с запасом в 1 час)
            if cache_min <= start_date and cache_max >= (end_date - timedelta(minutes=60)):
                print(f"[{ticker}] Данные полностью взяты из кэша.")
                mask = (df['DateTime'] >= start_date) & (df['DateTime'] <= end_date)
                return df.loc[mask].sort_values("DateTime", ascending=False)
            
            print(f"[{ticker}] Кэш неполный. Докачиваю недостающее...")

    # 3. Скачивание данных через API
    figi = get_figi(ticker) # Убедитесь, что функция get_figi определена
    new_rows = []
    
    # Для API Тинькофф нужны даты с часовым поясом (UTC)
    current_download = start_date.replace(tzinfo=timezone.utc)
    target_end = end_date.replace(tzinfo=timezone.utc)
    
    with Client(TOKEN) as client:
        while current_download < target_end:
            chunk_end = min(current_download + chunk_step, target_end)

            for attempt in range(3):
                try:
                    candles = client.market_data.get_candles(
                        figi=figi,
                        from_=current_download,
                        to=chunk_end,
                        interval=t_interval
                    ).candles

                    for c in candles:
                        new_rows.append({
                            "DateTime": c.time,
                            "Open":  c.open.units  + c.open.nano  / 1e9,
                            "High":  c.high.units  + c.high.nano  / 1e9,
                            "Low":   c.low.units   + c.low.nano   / 1e9,
                            "Close": c.close.units + c.close.nano / 1e9,
                            "Volume": c.volume
                        })
                    break 
                except Exception as e:
                    if "RESOURCE_EXHAUSTED" in str(e):
                        print("Лимит запросов! Жду 45 сек...")
                        time.sleep(45)
                    else:
                        print(f"Ошибка: {e}. Пробую снова...")
                        time.sleep(5)
            
            current_download = chunk_end
            time.sleep(0.5) # Небольшая пауза между чанками

    # 4. Объединение, очистка и сохранение
    if new_rows:
        new_df = pd.DataFrame(new_rows)
        # Сразу очищаем новые данные от TZ
        new_df['DateTime'] = pd.to_datetime(new_df['DateTime']).dt.tz_localize(None)
        
        df = pd.concat([df, new_df]).drop_duplicates(subset=['DateTime'])
    
    if df.empty:
        raise RuntimeError(f"Не удалось получить данные для {ticker}")

    # Сортируем и сохраняем полный кэш
    df = df.sort_values("DateTime", ascending=False)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    df.to_parquet(file_path, compression="zstd", index=False)
    
    # Возвращаем только запрошенный диапазон
    mask = (df['DateTime'] >= start_date) & (df['DateTime'] <= end_date)
    return df.loc[mask]

In [97]:
# Получить дневки для Газпрома
# df_GAZP = get_candles_data("GAZP", interval_name="1day", start_date=datetime(2023, 1, 1))

# Получить часовики для Сбера
# df_hourly = get_candles_data("SBER", interval_name="1hour")
df_SBER = get_candles_data("SBER", interval_name="1day", start_date=datetime(2015, 1, 1))
df_MOEX = get_candles_data("MOEX", interval_name="1day", start_date=datetime(2015, 1, 1))

[SBER] Кэш неполный. Докачиваю недостающее...
Подтверждено: SBER (Сбер Банк), FIGI: BBG004730N88
Подтверждено: MOEX (Московская Биржа), FIGI: BBG004730JJ5


In [98]:
df_SBER

,DateTime,Open,High,Low,Close,Volume
2874,2026-01-18,301.28,301.76,301.10,301.31,606490
0,2026-01-17,301.67,301.89,301.00,301.28,1061595
1,2026-01-16,299.43,301.97,298.90,301.52,20712588
2,2026-01-15,298.25,299.80,297.40,299.16,13250411
3,2026-01-14,297.23,299.78,296.02,298.58,18334454
...,...,...,...,...,...,...
2864,2015-01-07,56.00,58.45,55.83,58.28,70269700
2865,2015-01-06,56.00,58.45,55.83,58.28,70269700
2866,2015-01-05,54.02,56.77,53.58,56.37,63231040
2867,2015-01-02,53.04,55.65,51.60,54.90,120437150


In [99]:
def clean_market_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Очищает данные от пропусков и нулевых значений.
    """
    df = df.copy()
    
    # 1. Заменяем чистые нули на NaN (в цене Close/Open нулей быть не может)
    # Делаем это только для колонок с ценами
    cols_to_fix = ['Open', 'High', 'Low', 'Close']
    for col in cols_to_fix:
        df[col] = df[col].replace(0, pd.NA)
    
    # 2. Сортируем по времени (важно для правильного заполнения "вперед")
    df = df.sort_values("DateTime")
    
    # 3. Forward Fill: заполняем пропуски последним известным значением
    df[cols_to_fix] = df[cols_to_fix].ffill()
    
    # 4. Backward Fill: на случай, если пропуски в самых первых строках
    df[cols_to_fix] = df[cols_to_fix].bfill()
    
    # 5. Обработка объема (Volume)
    # Для объема пропуски лучше заменять на 0, так как сделок просто не было
    df['Volume'] = df['Volume'].fillna(0)
    
    return df

In [100]:
def fill_time_gaps(df: pd.DataFrame, interval_name: str) -> pd.DataFrame:
    """
    Вставляет пропущенные временные интервалы (строки), которых нет в данных.
    """
    # Маппинг для метода resample
    resample_map = {
        "5min": "5min", "15min": "15min", "1hour": "h", "1day": "D"
    }
    freq = resample_map.get(interval_name, "5min")
    
    df = df.set_index("DateTime").sort_index()
    
    # Создаем полный индекс без дыр от начала до конца
    full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq=freq)
    
    # Переиндексируем (появятся строки с NaN там, где были дыры)
    df = df.reindex(full_range)
    df.index.name = "DateTime"
    
    # Заполняем пустоты
    df[['Open', 'High', 'Low', 'Close']] = df[['Open', 'High', 'Low', 'Close']].ffill()
    df['Volume'] = df['Volume'].fillna(0)
    
    return df.reset_index()

In [101]:
def add_technical_indicators(df: pd.DataFrame, 
                             sma_periods: list = [20, 50, 200],
                             ema_periods: list = [12, 26],
                             rsi_period: int = 14,
                             macd_fast: int = 12,
                             macd_slow: int = 26,
                             macd_signal: int = 9) -> pd.DataFrame:
    """
    Добавляет технические индикаторы к датафрейму с OHLCV данными.
    
    Параметры:
        df: DataFrame с колонками ['DateTime', 'Open', 'High', 'Low', 'Close', 'Volume']
        sma_periods: список периодов для SMA
        ema_periods: список периодов для EMA
        rsi_period: период для RSI
        macd_fast, macd_slow, macd_signal: параметры MACD
    
    Возвращает:
        Тот же DataFrame с добавленными колонками индикаторов
    """
    df = df.copy()
    close = df['Close']
    
    # SMA (Simple Moving Average)
    for period in sma_periods:
        df[f'SMA_{period}'] = close.rolling(window=period).mean()
    
    # EMA (Exponential Moving Average)
    for period in ema_periods:
        df[f'EMA_{period}'] = close.ewm(span=period, adjust=False).mean()
    
    # RSI (Relative Strength Index)
    delta = close.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    
    avg_gain = gain.rolling(window=rsi_period, min_periods=rsi_period).mean()
    avg_loss = loss.rolling(window=rsi_period, min_periods=rsi_period).mean()
    
    # Сглаживание по методу Wilder's (аналог ema с alpha = 1/period)
    avg_gain = avg_gain.ewm(alpha=1/rsi_period, min_periods=rsi_period, adjust=False).mean()
    avg_loss = avg_loss.ewm(alpha=1/rsi_period, min_periods=rsi_period, adjust=False).mean()
    
    rs = avg_gain / avg_loss
    df[f'RSI_{rsi_period}'] = 100 - (100 / (1 + rs))
    
    # MACD
    ema_fast = close.ewm(span=macd_fast, adjust=False).mean()
    ema_slow = close.ewm(span=macd_slow, adjust=False).mean()
    df['MACD'] = ema_fast - ema_slow
    df['MACD_Signal'] = df['MACD'].ewm(span=macd_signal, adjust=False).mean()
    df['MACD_Hist'] = df['MACD'] - df['MACD_Signal']
    
    return df

In [102]:
# # Пример использования
# ticker = "LKOH"
# df = get_5min(ticker)

# # Добавляем индикаторы
# df_with_indicators = add_technical_indicators(df)

# # Теперь в df_with_indicators есть:
# # SMA_10, SMA_20, SMA_50
# # EMA_12, EMA_26
# # RSI_14
# # MACD, MACD_Signal, MACD_Hist

# print(df_with_indicators.tail())

Стратегия DCA (Dollar Cost Averaging)
Вы покупаете 10-го и 25-го числа. Если день выходной, мы берем ближайший предыдущий рабочий день

In [103]:
def benchmark_dca(df: pd.DataFrame, investment_per_step: float = 1000):
    # 1. Подготовка данных
    df = df.copy()
    # Убеждаемся, что DateTime в индексе и без часовых поясов
    if 'DateTime' in df.columns:
        df['DateTime'] = pd.to_datetime(df['DateTime']).dt.tz_localize(None)
        df = df.set_index('DateTime')
    else:
        df.index = pd.to_datetime(df.index).tz_localize(None)
        
    df = df.sort_index()
    
    # 2. Генерируем целевые даты (10 и 25 число)
    start_date = df.index.min().replace(hour=0, minute=0, second=0)
    end_date = df.index.max()
    all_days = pd.date_range(start=start_date, end=end_date, freq='D')
    target_days = all_days[(all_days.day == 10) | (all_days.day == 25)]
    
    total_shares = 0
    total_invested = 0
    
    # 3. Цикл покупок
    for day in target_days:
        # Ищем ближайший доступный торговый день (t, t-1, t-2...)
        # Используем asof, чтобы найти последнюю доступную цену до или в этот день
        try:
            # Находим индекс ближайшего торгового дня
            idx = df.index.get_indexer([day], method='pad')[0]
            
            if idx == -1: # Если до этой даты нет данных
                continue
                
            buy_price = df.iloc[idx]['Open']
            
            # Если цена некорректная (NaN или 0), пропускаем
            if pd.isna(buy_price) or buy_price <= 0:
                continue
                
            shares_bought = investment_per_step / buy_price
            total_shares += shares_bought
            total_invested += investment_per_step
            
        except Exception:
            continue
            
    # 4. Финальный расчет
    if total_invested == 0:
        return 0.0, 0.0, 0.0
        
    final_price = df['Close'].iloc[-1]
    portfolio_value = total_shares * final_price
    profit_pct = ((portfolio_value - total_invested) / total_invested) * 100
    
    return profit_pct, portfolio_value, total_invested

In [104]:
# Запуск
profit_pct, portfolio_value, total_invested = benchmark_dca(df_SBER)
print(f"Доходность: {profit_pct:.2f}%")
print(f"Стоимость портфеля: {portfolio_value:.2f}")
print(f"Всего инвестировано: {total_invested:.2f}")

Доходность: 70.60%
Стоимость портфеля: 452093.27
Всего инвестировано: 265000.00


In [105]:
profit_pct, portfolio_value, total_invested = benchmark_dca(df_MOEX)
print(f"Доходность: {profit_pct:.2f}%")
print(f"Стоимость портфеля: {portfolio_value:.2f}")
print(f"Всего инвестировано: {total_invested:.2f}")

Доходность: 50.35%
Стоимость портфеля: 398433.70
Всего инвестировано: 265000.00


Стратегия "Sell in May and Go Away"
Классическая поговорка гласит: «Продавай в мае и уходи, возвращайся в День всех святых (ноябрь)». Логика: Мы владеем активом только с 1 ноября по 30 апреля. В остальное время мы «в кэше» (деньги просто лежат)

In [106]:
def benchmark_sell_in_may(df: pd.DataFrame, initial_capital: float = 10000):
    df = df.copy()
    
    # 1. ПРОВЕРКА ИНДЕКСА (Важно!)
    # Если DateTime в колонках, а не в индексе
    if 'DateTime' in df.columns:
        df['DateTime'] = pd.to_datetime(df['DateTime']).dt.tz_localize(None)
        df = df.set_index('DateTime')
    # Если индекс не DatetimeIndex, пробуем конвертировать
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index).tz_localize(None)
    
    df = df.sort_index()
    
    # 2. РЕСЭМПЛИНГ (Теперь сработает без ошибок)
    daily = df.resample('D').agg({'Open': 'first', 'Close': 'last'}).dropna()
    
    in_market_months = [11, 12, 1, 2, 3, 4]
    current_cash = initial_capital
    shares = 0
    in_position = False

    for date, row in daily.iterrows():
        month = date.month
        
        # Вход
        if month in in_market_months and not in_position:
            shares = current_cash / row['Open']
            current_cash = 0
            in_position = True
            
        # Выход
        elif month not in in_market_months and in_position:
            current_cash = shares * row['Open']
            shares = 0
            in_position = False

    # Финальная стоимость
    final_value = current_cash if not in_position else shares * daily['Close'].iloc[-1]
    total_return = (final_value - initial_capital) / initial_capital * 100
    
    return initial_capital, final_value - initial_capital, total_return

In [107]:
initial_capital, final_value, total_return = benchmark_sell_in_may(df_SBER, initial_capital = 265000)
print(initial_capital, final_value, total_return)

265000 780689.7801840783 294.59991705059554


In [108]:
initial_capital, final_value, total_return = benchmark_sell_in_may(df_MOEX, initial_capital = 265000)
print(initial_capital, final_value, total_return)

265000 393987.2828737163 148.6744463674401


Стратегия с банковским вкладом

In [109]:
def benchmark_deposit(initial_capital, df_rates, start_year, end_year):
    """
    initial_capital: Стартовая сумма
    df_rates: DataFrame с колонкой 'MaxRate' и годами в индексе
    start_year: Год начала (int)
    end_year: Год окончания (int)
    """
    current_balance = initial_capital
    history = []
    
    # Добавляем стартовую точку (начало первого года)
    history.append({
        'Year': start_year,
        'Balance': current_balance,
        'Rate': 0,
        'Profit': 0
    })
    
    # Цикл по годам
    for year in range(start_year, end_year + 1):
        if year in df_rates.index:
            rate = df_rates.loc[year, 'MaxRate']
            
            # В конце года получаем выплату
            profit = current_balance * rate
            current_balance += profit
            
            history.append({
                'Year': year + 1, # Состояние на 1 января следующего года
                'Balance': round(current_balance, 2),
                'Rate': rate,
                'Profit': round(profit, 2)
            })
        else:
            print(f"Предупреждение: Ставка за {year} год не найдена.")
            
    # Превращаем историю в DataFrame для удобного построения графиков
    df_history = pd.DataFrame(history)
    
    total_return_pct = (current_balance - initial_capital) / initial_capital * 100
    return total_return_pct, current_balance, df_history

In [110]:
start_cap = 100000 # 100 тысяч рублей
ret_pct, final_val, history_df = benchmark_deposit(start_cap, df_rates, 2015, 2025)

print(f"Итоговая доходность вкладов: {ret_pct:.2f}%")
print(f"Финальная сумма: {final_val:.2f}")

# Посмотрим на историю роста
print(history_df)

Итоговая доходность вкладов: 185.68%
Финальная сумма: 285677.84
    Year    Balance     Rate    Profit
0   2015  100000.00  0.00000      0.00
1   2016  115325.00  0.15325  15325.00
2   2017  126845.97  0.09990  11520.97
3   2018  137501.03  0.08400  10655.06
4   2019  147462.98  0.07245   9961.95
5   2020  158565.47  0.07529  11102.49
6   2021  167963.64  0.05927   9398.18
7   2022  175498.49  0.04486   7534.85
8   2023  189078.56  0.07738  13580.07
9   2024  204450.65  0.08130  15372.09
10  2025  234695.04  0.14793  30244.38
11  2026  285677.84  0.21723  50982.80


In [111]:
import yfinance as yf

stock = yf.Ticker('SBER.ME')
stock.info
# tickers = ['SBER.ME', 'GAZP.ME', 'ROSN.ME', 'NVTK.ME', 'GMKN.ME',
#            'LKOH.ME', 'PLZL.ME', 'YNDX.ME', 'SNGS.ME', 'SNGSP.ME']  # Пример тикеров для MOEX

# data = {}
# for ticker in tickers:
#     stock = yf.Ticker(ticker)
#     info = stock.info
#     data[ticker] = info.get('marketCap', None)

# # Отсортировать по капитализации
# # sorted_tickers = sorted(data.items(), key=lambda x: x[1], reverse=True)
# for ticker, cap in data.items():
#     print(f"{ticker}: {cap}")

{'address1': '19 Vavilova Street',
 'city': 'Moscow',
 'zip': '117312',
 'country': 'Russia',
 'phone': '7 495 957 5731',
 'fax': '7 495 747 3731',
 'website': 'https://www.sberbank.ru',
 'industry': 'Banks - Regional',
 'industryKey': 'banks-regional',
 'industryDisp': 'Banks - Regional',
 'sector': 'Financial Services',
 'sectorKey': 'financial-services',
 'sectorDisp': 'Financial Services',
 'longBusinessSummary': 'Sberbank of Russia, together with its subsidiaries, engages in corporate and retail banking to corporate customers, large, medium, and small businesses, government bodies, and financial organizations in the Russian Federation. It operates through B2B (business to business); B2C (business to clients): Other segments.The company offers online banking; deposit and corporate structured products; ending products to corporate clients comprising lending, trade and export financing, interbank and overdraft lending, REPO, leasing, and other financing instruments; lending products 